# Clima

This notebook reads the EPW weather file and analyzes the air temperature series.

The simplest option here is `pvlib.iotools.read_epw`, which returns a pandas DataFrame plus metadata.

You can use the `temp_air` column for temperature analysis.

In [3]:
import plotly.graph_objects as go

from pvlib.iotools import read_epw

path = '../epw/MEX_CMX_Cuidad.Mexico.Central.766800_TMYx.2011-2025.epw'
weather, metadata = read_epw(path)

month_labels = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
monthly_temperature = [weather.loc[weather.index.month == month, 'temp_air'] for month in range(1, 13)]
monthly_mean = weather['temp_air'].groupby(weather.index.month).mean().reindex(range(1, 13))

fig = go.Figure()
for month_label, temperatures in zip(month_labels, monthly_temperature):
    fig.add_trace(go.Box(
        y=temperatures,
        name=month_label,
        boxpoints=False,
        marker_color='#219ebc',
        line_color='#1d3557',
    ))

fig.add_trace(go.Scatter(
    x=month_labels,
    y=monthly_mean.values,
    mode='lines+markers',
    name='Media mensual',
    line=dict(color='#e76f51', width=3),
    marker=dict(color='#e76f51', size=8),
))

fig.update_layout(
    width=1000,
    height=400,
    template='plotly_white',
    yaxis_title='To [°C]',
)

In [4]:
import pandas as pd

hdd_base_f = 65
cdd_base_f = 50
hdd_base_c = (hdd_base_f - 32) * 5 / 9
cdd_base_c = (cdd_base_f - 32) * 5 / 9

daily_mean = weather['temp_air'].resample('D').mean()

month_stats = pd.DataFrame({
    'mean_temp': weather['temp_air'].groupby(weather.index.month).mean(),
    'min_temp': weather['temp_air'].groupby(weather.index.month).min(),
    'max_temp': weather['temp_air'].groupby(weather.index.month).max(),
})
month_stats.index = [month_labels[i - 1] for i in month_stats.index]
month_stats['HDD18.3'] = (hdd_base_c - daily_mean).clip(lower=0).groupby(daily_mean.index.month).sum().values
month_stats['CDD10'] = (daily_mean - cdd_base_c).clip(lower=0).groupby(daily_mean.index.month).sum().values

month_stats = month_stats.round(2)
print(month_stats)
print()
print(f'Los Grados Día de Calefacción Anual base {hdd_base_c:.1f} °C (equivalente a {hdd_base_f} °F) reportados por CONUEE son 2351, y los Grados Día de Refrigeración Anual base {cdd_base_c:.0f} °C (equivalente a {cdd_base_f} °F) son 4959.')
print('Esto coincide con el análisis mensual que se muestra abajo, donde enero es el mes crítico frío y mayo es el mes crítico cálido.')

     mean_temp  min_temp  max_temp  HDD18.3   CDD10
Jan      12.21       2.4      24.4   189.89   68.44
Feb      14.67       3.7      27.0   102.61  130.72
Mar      16.93       6.5      28.9    45.67  214.91
Apr      18.31       8.1      30.1    21.98  249.34
May      18.70       8.3      31.6    16.87  269.79
Jun      17.93       9.0      28.3    26.30  237.94
Jul      17.20       8.9      26.3    35.22  223.19
Aug      16.91       8.8      25.8    44.30  214.36
Sep      16.88       9.6      26.0    43.64  206.47
Oct      15.54       6.0      28.6    86.75  171.59
Nov      13.86       0.1      25.4   134.27  119.68
Dec      12.91       3.0      25.2   168.05   90.78

Los Grados Día de Calefacción Anual base 18.3 °C (equivalente a 65 °F) reportados por CONUEE son 2351, y los Grados Día de Refrigeración Anual base 10 °C (equivalente a 50 °F) son 4959.
Esto coincide con el análisis mensual que se muestra abajo, donde enero es el mes crítico frío y mayo es el mes crítico cálido.


## Interpretation

Using the EPW monthly temperature statistics and degree-days with separate bases of 18.3 °C for heating and 10 °C for cooling:

- **Critical cold month:** **January**. It has the lowest mean temperature (12.21 °C) and the highest heating degree-days (HDD18.3 = 189.89).
- **Critical hot month:** **May**. It has the highest mean temperature (18.70 °C), the highest maximum temperature (31.6 °C), and the highest cooling degree-days (CDD10 = 269.79).
- **Secondary cold period:** **November to December**, which also show low mean temperatures and large HDD values.
- **Secondary warm period:** **April to June**, especially April and May, where both mean temperature and CDD rise sharply.

Bioclimatic conclusion: the design priority is to support **winter comfort and solar gain in the cold season** and **shading / overheating control in late spring and early summer**.

In [5]:
import pandas as pd
from pvlib.iotools import read_epw
f = "../epw/MEX_CMX_Cuidad.Mexico.Central.766800_TMYx.2011-2025.epw"
weather, metadata = read_epw(f)

gdc_base_f = 65
gdr_base_f = 50
gdc_base_c = (gdc_base_f - 32) * 5 / 9
gdr_base_c = (gdr_base_f - 32) * 5 / 9

media_diaria = weather['temp_air'].resample('D').mean()

estadisticas_mensuales = pd.DataFrame({
    'temp_media': weather['temp_air'].groupby(weather.index.month).mean(),
    'temp_min': weather['temp_air'].groupby(weather.index.month).min(),
    'temp_max': weather['temp_air'].groupby(weather.index.month).max(),
})
meses_es = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
estadisticas_mensuales.index = [meses_es[i - 1] for i in estadisticas_mensuales.index]
estadisticas_mensuales['GDCA18.3'] = (gdc_base_c - media_diaria).clip(lower=0).groupby(media_diaria.index.month).sum().values
estadisticas_mensuales['GDRA10'] = (media_diaria - gdr_base_c).clip(lower=0).groupby(media_diaria.index.month).sum().values
estadisticas_mensuales = estadisticas_mensuales.round(2)
print(estadisticas_mensuales)

     temp_media  temp_min  temp_max  GDCA18.3  GDRA10
Ene       12.21       2.4      24.4    189.89   68.44
Feb       14.67       3.7      27.0    102.61  130.72
Mar       16.93       6.5      28.9     45.67  214.91
Abr       18.31       8.1      30.1     21.98  249.34
May       18.70       8.3      31.6     16.87  269.79
Jun       17.93       9.0      28.3     26.30  237.94
Jul       17.20       8.9      26.3     35.22  223.19
Ago       16.91       8.8      25.8     44.30  214.36
Sep       16.88       9.6      26.0     43.64  206.47
Oct       15.54       6.0      28.6     86.75  171.59
Nov       13.86       0.1      25.4    134.27  119.68
Dic       12.91       3.0      25.2    168.05   90.78


In [22]:
weather

,year,month,day,hour,minute,data_source_unct,temp_air,temp_dew,relative_humidity,atmospheric_pressure,...,ceiling_height,present_weather_observation,present_weather_codes,precipitable_water,aerosol_optical_depth,snow_depth,days_since_last_snowfall,albedo,liquid_precipitation_depth,liquid_precipitation_quantity
2022-01-01 00:00:00-06:00,2022,1,1,1,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,7.4,4.6,82,76550,...,77777,9,999999999,13,0.085,0,88,0.13,0.0,0.0
2022-01-01 01:00:00-06:00,2022,1,1,2,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,6.4,4.2,86,76550,...,77777,9,999999999,12,0.085,0,88,0.13,0.0,0.0
2022-01-01 02:00:00-06:00,2022,1,1,3,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,5.9,3.5,85,76550,...,77777,9,999999999,11,0.085,0,88,0.13,0.0,0.0
2022-01-01 03:00:00-06:00,2022,1,1,4,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,4.7,3.3,91,76550,...,77777,9,999999999,11,0.085,0,88,0.13,0.0,0.0
2022-01-01 04:00:00-06:00,2022,1,1,5,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,4.3,2.8,90,76550,...,77777,9,999999999,11,0.085,0,88,0.13,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 19:00:00-06:00,2019,12,31,20,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,14.5,5.6,55,77719,...,77777,9,999999999,14,0.085,0,88,0.13,0.0,0.0
2019-12-31 20:00:00-06:00,2019,12,31,21,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,12.8,5.5,61,77719,...,2561,9,999999999,14,0.085,0,88,0.13,0.0,0.0
2019-12-31 21:00:00-06:00,2019,12,31,22,0,?9?9?9?9E0?9?9?9?9*9?9?9?9*9?9?9*9?9*9*9?9*9,12.5,5.7,63,77719,...,77777,9,999999999,14,0.085,0,88,0.13,0.0,0.0
2019-12-31 22:00:00-06:00,2019,12,31,23,0,?9?9?9?9E0?9?9?9*9*9?9?9?9*9?9?9*9?9*9*9?9*9,10.5,5.5,71,77719,...,77777,9,999999999,14,0.085,0,88,0.13,0.0,0.0


In [30]:
temps = weather[["temp_air"]].copy()
temps["hdd18_3"] = (18.3 - temps["temp_air"]).clip(lower=0)
temps["cdd10"] = (temps["temp_air"] - 10).clip(lower=0)
print(temps.head())

                           temp_air  hdd18_3  cdd10
2022-01-01 00:00:00-06:00       7.4     10.9    0.0
2022-01-01 01:00:00-06:00       6.4     11.9    0.0
2022-01-01 02:00:00-06:00       5.9     12.4    0.0
2022-01-01 03:00:00-06:00       4.7     13.6    0.0
2022-01-01 04:00:00-06:00       4.3     14.0    0.0


In [35]:
import numpy as np

daily_dd = pd.DataFrame({
    "hdd18_3": temps["hdd18_3"].resample('D').apply(lambda s: np.trapezoid(s.to_numpy(), dx=1) / 24),
    "cdd10": temps["cdd10"].resample('D').apply(lambda s: np.trapezoid(s.to_numpy(), dx=1) / 24),
})

yearly_dd = daily_dd.sum()

print(daily_dd.head())
print()
print(yearly_dd)

                            hdd18_3     cdd10
2011-02-01 00:00:00-06:00  5.325000  5.133333
2011-02-02 00:00:00-06:00  5.622917  4.504167
2011-02-03 00:00:00-06:00  4.481250  4.733333
2011-02-04 00:00:00-06:00  4.539583  4.441667
2011-02-05 00:00:00-06:00  2.985417  5.310417

hdd18_3    1241.389583
cdd10      2281.377083
dtype: float64


In [38]:
monthly_dd = daily_dd.groupby(daily_dd.index.month).sum()
monthly_dd.index = [month_labels[i - 1] for i in monthly_dd.index]

print(monthly_dd.round(3))
print()
print('Yearly totals:')
print(monthly_dd.sum().round(3))

     hdd18_3    cdd10
Jan  190.988  110.331
Feb  127.575  151.096
Mar  100.106  219.019
Apr   72.879  245.677
May   64.446  262.758
Jun   57.775  231.067
Jul   65.402  217.081
Aug   66.656  208.173
Sep   69.140  201.090
Oct  106.263  170.806
Nov  145.006  137.460
Dec  175.154  126.819

Yearly totals:
hdd18_3    1241.390
cdd10      2281.377
dtype: float64


In [40]:
import numpy as np
from scipy.integrate import simpson

def integrate_day(values, method):
    array = values.to_numpy()
    if array.size < 2:
        return 0.0
    if method == 'rectangle':
        return array.sum() / 24
    if method == 'trapezoid':
        return np.trapezoid(array, dx=1) / 24
    if method == 'simpson':
        return simpson(array, dx=1) / 24
    raise ValueError(method)

comparison_rows = []
for column in ["hdd18_3", "cdd10"]:
    hourly = temps[column]
    daily_groups = hourly.groupby(hourly.index.normalize())
    daily_rect = daily_groups.apply(lambda s: integrate_day(s, 'rectangle'))
    daily_trap = daily_groups.apply(lambda s: integrate_day(s, 'trapezoid'))
    daily_simp = daily_groups.apply(lambda s: integrate_day(s, 'simpson'))

    comparison_rows.append({
        'metric': column,
        'rectangle_year': daily_rect.sum(),
        'trapezoid_year': daily_trap.sum(),
        'simpson_year': daily_simp.sum(),
        'trap_minus_rect': daily_trap.sum() - daily_rect.sum(),
        'simp_minus_trap': daily_simp.sum() - daily_trap.sum(),
    })

comparison = pd.DataFrame(comparison_rows).set_index('metric')
print(comparison.round(6))

         rectangle_year  trapezoid_year  simpson_year  trap_minus_rect  \
metric                                                                   
hdd18_3     1323.308333     1241.389583   1240.954167        -81.91875   
cdd10       2331.470833     2281.377083   2279.105556        -50.09375   

         simp_minus_trap  
metric                    
hdd18_3        -0.435417  
cdd10          -2.271528  


In [41]:
daily_mean_temp = weather['temp_air'].resample('D').mean()

official_daily_dd = pd.DataFrame({
    'hdd18_3': (18.3 - daily_mean_temp).clip(lower=0),
    'cdd10': (daily_mean_temp - 10).clip(lower=0),
})
official_annual_dd = official_daily_dd.sum()
comparison_to_hourly = pd.Series({
    'official_hdd18_3': official_annual_dd['hdd18_3'],
    'official_cdd10': official_annual_dd['cdd10'],
    'hourly_hdd18_3': yearly_dd['hdd18_3'],
    'hourly_cdd10': yearly_dd['cdd10'],
    'hdd_difference_hourly_minus_official': yearly_dd['hdd18_3'] - official_annual_dd['hdd18_3'],
    'cdd_difference_hourly_minus_official': yearly_dd['cdd10'] - official_annual_dd['cdd10'],
})

print(official_daily_dd.head())
print()
print(comparison_to_hourly.round(3))

                            hdd18_3     cdd10
2011-02-01 00:00:00-06:00  4.287500  4.012500
2011-02-02 00:00:00-06:00  4.833333  3.466667
2011-02-03 00:00:00-06:00  4.345833  3.954167
2011-02-04 00:00:00-06:00  4.454167  3.845833
2011-02-05 00:00:00-06:00  2.879167  5.420833

official_hdd18_3                         905.138
official_cdd10                          2197.204
hourly_hdd18_3                          1241.390
hourly_cdd10                            2281.377
hdd_difference_hourly_minus_official     336.252
cdd_difference_hourly_minus_official      84.173
dtype: float64


In [ ]:
monthly_hourly_mean = weather['temp_air'].groupby([weather.index.month, weather.index.hour]).mean().unstack(level=1)
monthly_hourly_mean = monthly_hourly_mean.reindex(columns=range(24))

monthly_conuee_dd = pd.DataFrame({
    'HDD18.3': (18.3 - monthly_hourly_mean).clip(lower=0).sum(axis=1) / 24,
    'CDD10': (monthly_hourly_mean - 10).clip(lower=0).sum(axis=1) / 24,
})
monthly_conuee_dd.index = [month_labels[month - 1] for month in monthly_conuee_dd.index]

days_in_month = weather['temp_air'].groupby(weather.index.month).size().div(24)
annual_conuee_dd = monthly_conuee_dd.mul(days_in_month.values, axis=0).sum()

comparison_conuee = pd.DataFrame(index=['hdd18_3', 'cdd10'])
comparison_conuee['official_daily_mean'] = [official_annual_dd['hdd18_3'], official_annual_dd['cdd10']]
comparison_conuee['hourly_interval'] = [yearly_dd['hdd18_3'], yearly_dd['cdd10']]
comparison_conuee['conuee_monthly_avg_day'] = [annual_conuee_dd['HDD18.3'], annual_conuee_dd['CDD10']]
comparison_conuee['hourly_minus_official'] = comparison_conuee['hourly_interval'] - comparison_conuee['official_daily_mean']
comparison_conuee['conuee_minus_official'] = comparison_conuee['conuee_monthly_avg_day'] - comparison_conuee['official_daily_mean']
comparison_conuee['conuee_minus_hourly'] = comparison_conuee['conuee_monthly_avg_day'] - comparison_conuee['hourly_interval']

print('Monthly average-day degree days by CONUEE-style method:')
print(monthly_conuee_dd.round(3))
print()
print('Annual totals weighted by days in month:')
print(annual_conuee_dd.round(3))
print()
print('Comparison against the previous methºods:')
print(comparison_conuee.round(3))

Monthly average-day degree days by CONUEE-style method:
     HDD18.3  CDD10
Jan    6.397  3.506
Feb    4.766  5.341
Mar    3.306  7.156
Apr    2.328  8.311
May    1.884  8.703
Jun    1.732  7.931
Jul    2.117  7.200
Aug    2.178  6.915
Sep    2.364  6.882
Oct    3.549  5.535
Nov    4.919  4.327
Dec    5.853  4.031

Annual totals weighted by days in month:
HDD18.3    1257.554
CDD10      2307.529
dtype: float64

Comparison against the previous methods:
         official_daily_mean  hourly_interval  conuee_monthly_avg_day  \
hdd18_3              905.138         1241.390                1257.554   
cdd10               2197.204         2281.377                2307.529   

         hourly_minus_official  conuee_minus_official  conuee_minus_hourly  
hdd18_3                336.252                352.417               16.165  
cdd10                   84.173                110.325               26.152  


In [50]:
monthly_hourly_mean

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
1,8.135484,7.264516,6.522581,5.648387,5.458065,5.516129,5.345161,6.832258,9.583871,11.477419,...,21.354839,19.070968,18.406452,17.567742,15.767742,12.941935,11.829032,11.138710,9.932258,8.593548
2,10.364286,9.125000,8.400000,7.167857,7.003571,7.021429,6.514286,8.628571,11.639286,12.964286,...,23.278571,22.610714,22.089286,21.500000,19.767857,17.535714,15.717857,14.710714,12.085714,11.235714
3,12.964516,11.393548,10.570968,8.706452,8.561290,8.906452,8.461290,12.648387,15.651613,16.796774,...,25.654839,24.509677,23.790323,23.077419,21.503226,18.603226,17.648387,17.058065,14.000000,13.496774
4,14.276667,12.316667,11.876667,10.473333,10.310000,10.430000,10.290000,16.400000,18.246667,19.430000,...,26.306667,24.806667,24.076667,23.196667,21.646667,19.663333,18.540000,18.276667,15.953333,14.880000
5,15.874194,14.116129,13.077419,10.896774,10.812903,11.132258,11.248387,17.535484,18.877419,20.367742,...,25.345161,24.503226,23.706452,23.038710,21.929032,20.412903,19.325806,18.548387,16.738710,16.351613
6,15.416667,14.440000,13.760000,12.820000,12.870000,13.166667,12.560000,16.376667,17.330000,18.830000,...,23.353333,22.253333,21.640000,21.140000,20.140000,19.073333,17.850000,17.406667,16.343333,15.990000
7,15.019355,14.148387,13.606452,12.441935,12.558065,12.925806,12.500000,15.400000,16.406452,18.019355,...,23.035484,21.790323,20.554839,19.312903,18.235484,17.148387,16.493548,16.367742,15.490323,15.222581
8,14.896774,14.116129,13.658065,12.483871,12.470968,12.848387,12.364516,15.170968,16.090323,17.625806,...,22.322581,21.148387,19.980645,18.800000,18.000000,17.377419,16.612903,16.219355,15.551613,15.035484
9,14.176667,13.706667,13.383333,12.243333,12.270000,13.140000,12.800000,15.413333,16.463333,17.596667,...,23.003333,21.163333,19.973333,18.483333,17.596667,16.256667,16.040000,15.366667,15.073333,14.533333
10,12.470968,11.900000,11.148387,10.087097,10.170968,10.538710,10.348387,13.712903,15.551613,16.422581,...,22.925806,19.380645,19.016129,17.900000,16.674194,14.932258,14.377419,14.051613,13.122581,12.503226


In [51]:
monthly_conuee_dd

,HDD18.3,CDD10
1,6.396505,3.506452
2,4.766071,5.341071
3,3.305645,7.156183
4,2.327917,8.311389
5,1.884005,8.702823
6,1.732083,7.931389
7,2.117339,7.199597
8,2.178226,6.914785
9,2.364167,6.882222
10,3.549462,5.535081


In [44]:
conuee_12_mean_day_total = monthly_conuee_dd.sum()

method_comparison = pd.DataFrame(index=['hdd18_3', 'cdd10'])
method_comparison['official_daily_mean'] = [official_annual_dd['hdd18_3'], official_annual_dd['cdd10']]
method_comparison['hourly_interval'] = [yearly_dd['hdd18_3'], yearly_dd['cdd10']]
method_comparison['conuee_12_mean_days'] = [conuee_12_mean_day_total['HDD18.3'], conuee_12_mean_day_total['CDD10']]
method_comparison['hourly_minus_official'] = method_comparison['hourly_interval'] - method_comparison['official_daily_mean']
method_comparison['conuee_minus_official'] = method_comparison['conuee_12_mean_days'] - method_comparison['official_daily_mean']
method_comparison['conuee_minus_hourly'] = method_comparison['conuee_12_mean_days'] - method_comparison['hourly_interval']

print('CONUEE monthly mean-day method without day-of-month weighting:')
print(monthly_conuee_dd.round(3))
print()
print('Annual total from summing the 12 mean days:')
print(conuee_12_mean_day_total.round(3))
print()
print('Method comparison:')
print(method_comparison.round(3))
print()
print('Interpretation:')
print('1) official_daily_mean matches the usual daily-average degree-day definition.')
print('2) hourly_interval integrates the hourly excess temperatures within each day; it is a higher-resolution variant of the same idea when hourly data are available.')
print('3) conuee_12_mean_days applies the threshold to the 12 monthly average days and sums those 12 values, without weighting by the number of days in each month.')

CONUEE monthly mean-day method without day-of-month weighting:
     HDD18.3  CDD10
Jan    6.397  3.506
Feb    4.766  5.341
Mar    3.306  7.156
Apr    2.328  8.311
May    1.884  8.703
Jun    1.732  7.931
Jul    2.117  7.200
Aug    2.178  6.915
Sep    2.364  6.882
Oct    3.549  5.535
Nov    4.919  4.327
Dec    5.853  4.031

Annual total from summing the 12 mean days:
HDD18.3    41.393
CDD10      75.839
dtype: float64

Method comparison:
         official_daily_mean  hourly_interval  conuee_12_mean_days  \
hdd18_3              905.138         1241.390               41.393   
cdd10               2197.204         2281.377               75.839   

         hourly_minus_official  conuee_minus_official  conuee_minus_hourly  
hdd18_3                336.252               -863.744            -1199.996  
cdd10                   84.173              -2121.365            -2205.538  

Interpretation:
1) official_daily_mean matches the usual daily-average degree-day definition.
2) hourly_interval integ

In [45]:
method_summary = pd.DataFrame(index=['hdd18_3', 'cdd10'])
method_summary['official_daily_mean'] = [official_annual_dd['hdd18_3'], official_annual_dd['cdd10']]
method_summary['hourly_interval'] = [yearly_dd['hdd18_3'], yearly_dd['cdd10']]
method_summary['conuee_monthly_avg_day'] = [annual_conuee_dd['HDD18.3'], annual_conuee_dd['CDD10']]
method_summary['hourly_minus_official'] = method_summary['hourly_interval'] - method_summary['official_daily_mean']
method_summary['conuee_minus_official'] = method_summary['conuee_monthly_avg_day'] - method_summary['official_daily_mean']
method_summary['conuee_minus_hourly'] = method_summary['conuee_monthly_avg_day'] - method_summary['hourly_interval']

method_summary = method_summary.round(3)

print('Three meaningful degree-day methods computed from previous cells:')
print(method_summary)
print()
print('Notes:')
print('official_daily_mean uses the daily mean temperature threshold definition.')
print('hourly_interval uses the hourly integral form, equivalent to 1/24 of the accumulated hourly excess over each day.')
print('conuee_monthly_avg_day uses the 12 monthly average days, then weights each month by its number of days.')

Three meaningful degree-day methods computed from previous cells:
         official_daily_mean  hourly_interval  conuee_monthly_avg_day  \
hdd18_3              905.138         1241.390                1257.554   
cdd10               2197.204         2281.377                2307.529   

         hourly_minus_official  conuee_minus_official  conuee_minus_hourly  
hdd18_3                336.252                352.417               16.165  
cdd10                   84.173                110.325               26.152  

Notes:
official_daily_mean uses the daily mean temperature threshold definition.
hourly_interval uses the hourly integral form, equivalent to 1/24 of the accumulated hourly excess over each day.
conuee_monthly_avg_day uses the 12 monthly average days, then weights each month by its number of days.


In [7]:
3.38*2.91

9.8358

In [8]:
1.27*0.72

0.9144

In [9]:
9.4302+0.9144

10.3446

In [10]:
3.38-2.05

1.33

In [11]:
.12+1.15+1.99+.12-2.17

1.21

In [12]:
1.15+.12

1.27

In [13]:
0.72+1.95

2.67

In [47]:
f = "../epw/MEX_CMX_Cuidad.Mexico.Central.766800_TMYx.2011-2025.epw"
weather, metadata = read_epw(f)

gdc_base_f = 65
gdr_base_f = 50
gdc_base_c = (gdc_base_f - 32) * 5 / 9
gdr_base_c = (gdr_base_f - 32) * 5 / 9

media_diaria = weather['temp_air'].resample('D').mean()

estadisticas_mensuales = pd.DataFrame({
    'temp_media': weather['temp_air'].groupby(weather.index.month).mean(),
    'temp_min': weather['temp_air'].groupby(weather.index.month).min(),
    'temp_max': weather['temp_air'].groupby(weather.index.month).max(),
})
meses_es = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
estadisticas_mensuales.index = [meses_es[i - 1] for i in estadisticas_mensuales.index]
estadisticas_mensuales['GDCA18.3'] = (gdc_base_c - media_diaria).clip(lower=0).groupby(media_diaria.index.month).sum().values
estadisticas_mensuales['GDRA10'] = (media_diaria - gdr_base_c).clip(lower=0).groupby(media_diaria.index.month).sum().values
estadisticas_mensuales = estadisticas_mensuales.round(2)

In [54]:
f = "../epw/MEX_CMX_Cuidad.Mexico.Central.766800_TMYx.2011-2025.epw"
weather, metadata = read_epw(f)

gdc_base_f = 65
gdr_base_f = 50
gdc_base_c = (gdc_base_f - 32) * 5 / 9
gdr_base_c = (gdr_base_f - 32) * 5 / 9

estadisticas_mensuales = pd.DataFrame({
    'temp_media': weather['temp_air'].groupby(weather.index.month).mean(),
    'temp_min': weather['temp_air'].groupby(weather.index.month).min(),
    'temp_max': weather['temp_air'].groupby(weather.index.month).max(),
})
meses_es = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
estadisticas_mensuales.index = [meses_es[i - 1] for i in estadisticas_mensuales.index]

# Calcular GDC/GDR con CONUEE horas de los dias promedio mensuales y convertir a totales mensuales
monthly_hourly_mean = weather['temp_air'].groupby([weather.index.month, weather.index.hour]).mean().unstack(level=1)
monthly_hourly_mean = monthly_hourly_mean.reindex(columns=range(24))

monthly_conuee_dd = pd.DataFrame({
    'HDD18.3': (18.3 - monthly_hourly_mean).clip(lower=0).sum(axis=1) / 24,
    'CDD10': (monthly_hourly_mean - 10).clip(lower=0).sum(axis=1) / 24,
})

days_in_month = weather['temp_air'].groupby(weather.index.month).size().div(24)
monthly_totals = monthly_conuee_dd.mul(days_in_month.values, axis=0)
monthly_totals.index = [meses_es[m - 1] for m in monthly_totals.index]

estadisticas_mensuales['GDCA18.3'] = monthly_totals['HDD18.3'].values
estadisticas_mensuales['GDRA10'] = monthly_totals['CDD10'].values
estadisticas_mensuales = estadisticas_mensuales.round(2)
print(estadisticas_mensuales)

     temp_media  temp_min  temp_max  GDCA18.3  GDRA10
Ene       12.21       2.4      24.4    198.29  108.70
Feb       14.67       3.7      27.0    133.45  149.55
Mar       16.93       6.5      28.9    102.48  221.84
Abr       18.31       8.1      30.1     69.84  249.34
May       18.70       8.3      31.6     58.40  269.79
Jun       17.93       9.0      28.3     51.96  237.94
Jul       17.20       8.9      26.3     65.64  223.19
Ago       16.91       8.8      25.8     67.53  214.36
Sep       16.88       9.6      26.0     70.93  206.47
Oct       15.54       6.0      28.6    110.03  171.59
Nov       13.86       0.1      25.4    147.56  129.82
Dic       12.91       3.0      25.2    181.45  124.95


In [49]:
print(estadisticas_mensuales["GDCA18.3"].sum(), estadisticas_mensuales["GDRA10"].sum())

1257.56 2307.54


In [52]:
monthly_totals

,HDD18.3,CDD10
Ene,198.291667,108.700000
Feb,133.450000,149.550000
Mar,102.475000,221.841667
Abr,69.837500,249.341667
May,58.404167,269.787500
Jun,51.962500,237.941667
Jul,65.637500,223.187500
Ago,67.525000,214.358333
Sep,70.925000,206.466667
Oct,110.033333,171.587500
